# Round 10 — Lexical–semantic agreement

Two full independent families, frozen before execution. Cached vectors only; no model download, no new neural inference. This is exploratory development, not a Kaggle score.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "configs/crossmodal_support_features.json").is_file())
from scripts.run_crossmodal_support_features import figures, write_dashboard
SPEC = json.loads((ROOT / "configs/crossmodal_support_features.json").read_text())
RESULT = json.loads((ROOT / "reports/crossmodal_support_features/results.json").read_text())
print("Registered primary:", SPEC["primary"])
print("New classifier fits:", RESULT["new_fits"])
print("Prior readouts reused:", RESULT["reused_prior_controls"])
CHARTS = figures(RESULT)

Registered primary: crossmodal_all
New classifier fits: 12
Prior readouts reused: 10


## 1. Hypothesis and matched evidence

Does agreement between exact-word/character overlap and frozen semantic neighbors add evidence beyond the existing context-conditioned anchor?

The anchor was selected after Round 9 and is not promoted. All candidates retain the exact same saved anchor features and classifier.

In [2]:
display(pd.DataFrame(RESULT["metrics"]))
CHARTS[0].show(renderer="plotly_mimetype")

,fold,policy,variant,auc,brier,log_loss
0,0,"No Advertising: Spam, referral links, unsolici...",qwen_raw,0.679254,0.250238,0.760003
1,1,No legal advice: Do not offer or request legal...,qwen_raw,0.760533,0.221528,0.709879
2,0,"No Advertising: Spam, referral links, unsolici...",answer_only,0.679254,0.260747,0.828707
3,1,No legal advice: Do not offer or request legal...,answer_only,0.760533,0.228357,0.767258
4,0,"No Advertising: Spam, referral links, unsolici...",frozen_basic,0.693097,0.254469,0.827292
5,1,No legal advice: Do not offer or request legal...,frozen_basic,0.753038,0.234327,0.821515
6,0,"No Advertising: Spam, referral links, unsolici...",context_evidence,0.703470,0.247081,0.823188
7,1,No legal advice: Do not offer or request legal...,context_evidence,0.755044,0.235380,0.840367
8,0,"No Advertising: Spam, referral links, unsolici...",uniform_all,0.703246,0.247805,0.821082
9,1,No legal advice: Do not offer or request legal...,uniform_all,0.754574,0.233871,0.817539


## 2. Conditional uncertainty

A mean improvement can hide a policy regression. Paired intervals cover this round, not all adaptive research choices or future policies.

In [3]:
display(pd.DataFrame(RESULT["comparisons"]))
CHARTS[1].show(renderer="plotly_mimetype")

,candidate,reference,comparison,delta_auc,simultaneous_low,simultaneous_high
0,word_agreement,context_evidence,word_agreement vs context anchor,-0.009100,-0.034254,0.016054
1,character_agreement,context_evidence,character_agreement vs context anchor,-0.007147,-0.032301,0.018006
2,crossmodal_all,context_evidence,crossmodal_all vs context anchor,-0.012608,-0.037762,0.012546
3,alignment_null,context_evidence,alignment_null vs context anchor,-0.012952,-0.038106,0.012201
4,label_null,context_evidence,label_null vs context anchor,-0.009074,-0.034227,0.016080
5,lexical_only,context_evidence,lexical_only vs context anchor,-0.006749,-0.031903,0.018404
6,crossmodal_all,qwen_raw,Primary vs qwen_raw,-0.003244,-0.028398,0.021909
7,crossmodal_all,frozen_basic,Primary vs frozen_basic,-0.006419,-0.031572,0.018735
8,crossmodal_all,uniform_all,Primary vs uniform_all,-0.012261,-0.037415,0.012893
9,crossmodal_all,alignment_null,Primary vs alignment_null,0.000344,-0.024809,0.025498


## 3. Mechanism-specific controls

alignment_null; label_null; lexical_only. Controls diagnose attribution; no secondary winner may silently replace the primary.

In [4]:
CHARTS[2].show(renderer="plotly_mimetype")
CHARTS[3].show(renderer="plotly_mimetype")

## 4. Reference coverage and failure modes

Exactly 48 new columns are proposed. Vocabulary or class-mode limitations are reported, not hidden. No query labels enter these descriptors.

In [5]:
display(pd.DataFrame(RESULT["diagnostics"]))
CHARTS[4].show(renderer="plotly_mimetype")
CHARTS[5].show(renderer="plotly_mimetype")

,fold,inner_fold,self_overlap,family,vocabulary_columns,zero_lexical_query_fraction,reference_rows,query_rows,mean_top5_agreement,mean_lexical_nearest
0,0,0,0,word,6000,0.0,419,210,0.106190,0.254956
1,0,0,0,character,6000,0.0,419,210,0.115714,0.333906
2,0,1,0,word,6000,0.0,419,210,0.116667,0.269922
3,0,1,0,character,6000,0.0,419,210,0.110000,0.344791
4,0,2,0,word,6000,0.0,420,209,0.114354,0.263699
5,0,2,0,character,6000,0.0,420,209,0.113876,0.334316
6,0,outer_query,0,word,6000,0.0,629,234,0.059829,0.242189
7,0,outer_query,0,character,6000,0.0,629,234,0.066667,0.295867
8,1,0,0,word,6000,0.0,244,122,0.056557,0.145858
9,1,0,0,character,6000,0.0,244,122,0.068033,0.218603


## 5. Fitted associations, not causal importance

These coefficients summarize the trained linear readout. Matched ablations, not coefficient magnitude, determine evidence for a family.

In [6]:
CHARTS[6].show(renderer="plotly_mimetype")

## 6. Probability quality and metric definitions

Macro policy AUC, raw pooled AUC and within-policy-ranked pooled AUC are different diagnostics. None is a new hidden evaluation score.

In [7]:
display(pd.DataFrame(RESULT["pooled_metrics"]))
CHARTS[7].show(renderer="plotly_mimetype")

,variant,macro_auc,pooled_auc,ranked_pooled_auc
0,qwen_raw,0.719893,0.739666,0.738959
1,answer_only,0.719893,0.738680,0.738959
2,frozen_basic,0.723068,0.733852,0.737090
3,context_evidence,0.729257,0.736183,0.741330
4,uniform_all,0.728910,0.735914,0.740913
5,word_agreement,0.720157,0.732201,0.734321
6,character_agreement,0.722110,0.734237,0.736694
7,crossmodal_all,0.716649,0.733293,0.733878
8,alignment_null,0.716305,0.730814,0.731247
9,label_null,0.720183,0.729633,0.731215


## 7. Preregistered decision and remaining gaps

Primary must beat every specified comparator, without a policy regression. Adapted support-training answer margins remain in-sample: feature cross-fitting does not fix encoder-level dependence. The companion round is independent.

Research motivation: https://aclanthology.org/2021.acl-long.316/

In [8]:
print("Decision:", RESULT["decision"])
for item in RESULT["primary_requirements"]:
    print(item["reference"], item["per_policy_delta"], "passed:", item["passed"])
for limitation in RESULT["limitations"]:
    print(limitation)
print("Interactive dashboard:", write_dashboard(ROOT, RESULT))

Decision: DO_NOT_PROMOTE_PRIMARY
qwen_raw [0.0006343283582088688, -0.007123148274984725] passed: False
frozen_basic [-0.01320895522388077, 0.0003718126846832259] passed: False
context_evidence [-0.02358208955223884, -0.001634018903739598] passed: False
uniform_all [-0.02335820895522389, -0.0011643607757185759] passed: False
alignment_null [-0.004477611940298498, 0.005166239408230688] passed: False
label_null [-0.016716417910447756, 0.00964756071309747] passed: False
lexical_only [-0.010298507462686679, -0.0014187589283967128] passed: False
Exploratory same-cohort follow-up; no independent holdout or Kaggle score.
Both new designs were frozen before either new result; they are independent.
The context_evidence anchor was selected after Round 9 and was not promoted.
Reference features are group cross-fitted, but the adapted answer margin is in-sample.
Cross-fitting new features does not remove the inherited answer-score limitation.
Intervals cover planned within-round contrasts only, not